In [1]:
# Imports
import sys
from pathlib import Path
BASE = Path.cwd().parent.parent
sys.path.insert(0, str(BASE)) # Go to base directory


import pandas as pd

from src.data import Data
import string
from itertools import chain
from collections import Counter
import nltk
from nltk.corpus import stopwords
import re
from sklearn.feature_extraction.text import TfidfVectorizer
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import chromadb
import numpy as np
from tqdm import tqdm


d:\venv\mm-prod-search-and-reco-venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Get Data

d1 = Data()
d_train = d1.get_complete_data('train')
d_val = d1.get_complete_data('validation')

# Clean data
cols = [
    "product_title",
    "product_description",
    "product_bullet_point",
    "product_brand",
    "product_color"
]

d_clean_train = d_train
d_clean_val = d_val

for col in cols:
    # Remove URLs
    d_clean_train[col] = d_clean_train[col].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
    d_clean_val[col] = d_clean_val[col].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
    # Remove HTML tags
    d_clean_train[col] = d_clean_train[col].apply(lambda x: re.sub(r"<.*?>","", x))
    d_clean_val[col] = d_clean_val[col].apply(lambda x: re.sub(r"<.*?>","", x))
    # lowercase and remove punctuation
    d_clean_train[col]  = d_clean_train[col].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()
    d_clean_val[col]  = d_clean_val[col].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

d_clean_train["query"] = d_clean_train["query"].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
d_clean_train["query"] = d_clean_train["query"].apply(lambda x: re.sub(r"<.*?>","", x))
d_clean_train["query"]  = d_clean_train["query"].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

d_clean_val["query"] = d_clean_val["query"].apply(lambda x: re.sub(r"http\S+|www\S+|https\S+","", x))
d_clean_val["query"] = d_clean_val["query"].apply(lambda x: re.sub(r"<.*?>","", x))
d_clean_val["query"]  = d_clean_val["query"].str.lower().str.translate(str.maketrans("", "", string.punctuation)).str.split()

## Get list of docs for prods and queries
train_docs = d_clean_train[cols].apply(lambda x: " ".join(chain.from_iterable(x)), axis= 1)
train_query = d_clean_train["query"].apply(lambda x: " ".join(x))

val_docs = d_clean_val[cols].apply(lambda x: " ".join(chain.from_iterable(x)), axis= 1)
val_query = d_clean_val["query"].apply(lambda x: " ".join(x))

Loading ESIC DATA...
ESIC Downloaded succesfully!
Loading SQID DATA...
SQID Downloaded succesfully!
Loading Amazon Review DATA...
Amazon Review Data Downloaded succesfully!


In [4]:
nltk.download('stopwords')
# Get the top 10k words from the entire corpus consisting of product_title	product_description	product_bullet_point	product_brand	product_color:
## Just for checking what words are being used
stop_words = set(stopwords.words('english'))

corpus = (chain.from_iterable(chain.from_iterable(d_clean_train[col] for col in cols)))
filtered_corpus = [word for word in corpus if word not in stop_words]

word_freq = Counter(filtered_corpus)

top_10k = word_freq.most_common(10_000)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rahul\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [5]:
## TF-IDF feature vector of top 10K terms in corpus
tfidf = TfidfVectorizer(max_features=10_000, min_df=2)

prod_train_tfidf = tfidf.fit_transform(train_docs)
query_train_tfidf = tfidf.transform(train_query)

prod_val_tfidf = tfidf.transform(val_docs)
query_val_tfidf = tfidf.transform(val_query)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cpu')

In [7]:
## Write boilerplate for MLP

class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, output_dims, activation):
        super().__init__()
        layers = []

        prev_dim = input_dim
        for dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, dim))
            layers.append(activation())
            prev_dim = dim

        layers.append(nn.Linear(prev_dim, output_dims))

        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

In [8]:
class QueryProductDataset(Dataset):
    def __init__(self, queries, products, labels):
        self.queries = queries
        self.products = products
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        query = torch.tensor(
            self.queries[idx].toarray().squeeze(),
            dtype=torch.float32
        )

        product = torch.tensor(
            self.products[idx].toarray().squeeze(),
            dtype=torch.float32
        )

        label = torch.tensor(
            self.labels[idx],
            dtype=torch.float32
        )

        return query, product, label

In [9]:
def train_model(model, train_data_loader, val_data_loader, optimizer, loss_criterion, epochs=10, device=device):
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        running_train_loss = 0
        for query_x, prod_x, labels in train_data_loader:
            query_x = query_x.to(device)
            prod_x = prod_x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()

            query_emb = model(query_x)
            prod_emb = model(prod_x)

            loss = loss_criterion(query_emb, prod_emb, labels)
            loss.backward()

            optimizer.step()

            running_train_loss += (loss.item() * query_x.size(0))
        epoch_train_loss = running_train_loss / len(train_data_loader.dataset)
        train_losses.append(epoch_train_loss)

        model.eval()
        running_val_loss = 0
        with torch.no_grad():
            for query_x, prod_x, labels in val_data_loader:
                query_x = query_x.to(device)
                prod_x = prod_x.to(device)
                labels = labels.to(device)
                
                query_emb = model(query_x)
                prod_emb = model(prod_x)

                loss = loss_criterion(query_emb, prod_emb, labels)

                running_val_loss += (loss.item() * query_x.size(0))
        epoch_val_loss = running_val_loss / len(val_data_loader.dataset)
        val_losses.append(epoch_val_loss)

        print(f"Epoch [{epoch+1}/{epochs}]"
            f"Train Loss: {epoch_train_loss:.4f}"
            f"Val Loss: {epoch_val_loss:.4f}")

    return train_losses, val_losses

In [11]:
label_map = {"E":  1.0, "S":  1.0, "C":  1.0, "I": -1.0}
train_esci_labels = d_clean_train["esci_label"].map(label_map).to_numpy()
val_esci_labels = d_clean_val["esci_label"].map(label_map).to_numpy()

train_dataset = QueryProductDataset(query_train_tfidf, prod_train_tfidf, train_esci_labels)
val_dataset = QueryProductDataset(query_val_tfidf, prod_val_tfidf, val_esci_labels)

In [12]:
BATCH_SIZE = 128

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

In [ ]:
INPUT_DIM = 10_000
OUTPUT_DIM = 512
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

hidden_dim = [2048]
activation_fn = nn.ReLU

model = MLP(INPUT_DIM, hidden_dim, OUTPUT_DIM, activation_fn).to(device)
loss = nn.CosineEmbeddingLoss(0.2)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

Using device: cpu


In [14]:
train_losses, val_losses = train_model(model, train_loader, val_loader, optimizer, loss)

Epoch [1/10]Train Loss: 0.0861Val Loss: 0.0713
Epoch [2/10]Train Loss: 0.0798Val Loss: 0.0710
Epoch [3/10]Train Loss: 0.0743Val Loss: 0.0745
Epoch [4/10]Train Loss: 0.0674Val Loss: 0.0793
Epoch [5/10]Train Loss: 0.0600Val Loss: 0.0833
Epoch [6/10]Train Loss: 0.0539Val Loss: 0.0897
Epoch [7/10]Train Loss: 0.0489Val Loss: 0.0999
Epoch [8/10]Train Loss: 0.0447Val Loss: 0.0980
Epoch [9/10]Train Loss: 0.0412Val Loss: 0.1000
Epoch [10/10]Train Loss: 0.0386Val Loss: 0.1156


In [15]:
## Define search space for each hyperparameter

hidden_dim_space = [[2048], [1024], [4096, 1024]]
activation_fn_space = [nn.ReLU, nn.GELU, nn.LeakyReLU, nn.Sigmoid, nn.Tanh]
loss_space = [nn.CosineEmbeddingLoss(0.2), nn.CosineEmbeddingLoss(0.1), nn.CosineEmbeddingLoss(0.0)]

In [ ]:
for hidden_dim in hidden_dim_space:
    for activation_fn in activation_fn_space:
        for loss in loss_space:
            
            model = MLP(INPUT_DIM, hidden_dim, OUTPUT_DIM, activation_fn).to(device)
            optimizer = optim.Adam(model.parameters(), lr=1e-4)

            print(f"=" * 70)
            print(f"hidden_dim={hidden_dim} | activation={activation_fn.__name__} | loss={loss.__class__.__name__}({loss.margin})")
            print(f"=" * 70)
            train_losses, val_losses = train_model(model, train_loader, val_loader, optimizer, loss)

hidden_dim=[2048] | activation=ReLU | loss=CosineEmbeddingLoss(0.2)
Epoch [1/10]Train Loss: 0.0863Val Loss: 0.0716
Epoch [2/10]Train Loss: 0.0798Val Loss: 0.0711
Epoch [3/10]Train Loss: 0.0746Val Loss: 0.0733
Epoch [4/10]Train Loss: 0.0673Val Loss: 0.0774
Epoch [5/10]Train Loss: 0.0596Val Loss: 0.0886
Epoch [6/10]Train Loss: 0.0536Val Loss: 0.0889
Epoch [7/10]Train Loss: 0.0484Val Loss: 0.0984
Epoch [8/10]Train Loss: 0.0443Val Loss: 0.1060
Epoch [9/10]Train Loss: 0.0410Val Loss: 0.1060
Epoch [10/10]Train Loss: 0.0383Val Loss: 0.1037
hidden_dim=[2048] | activation=ReLU | loss=CosineEmbeddingLoss(0.1)
Epoch [1/10]Train Loss: 0.0965Val Loss: 0.0806
Epoch [2/10]Train Loss: 0.0899Val Loss: 0.0810
Epoch [3/10]Train Loss: 0.0853Val Loss: 0.0824
Epoch [4/10]Train Loss: 0.0764Val Loss: 0.0888
Epoch [5/10]Train Loss: 0.0679Val Loss: 0.0991
Epoch [6/10]Train Loss: 0.0605Val Loss: 0.1038
Epoch [7/10]Train Loss: 0.0541Val Loss: 0.1148
Epoch [8/10]Train Loss: 0.0492Val Loss: 0.1123
Epoch [9/10]Train

In [ ]:
BASE_DATA_DIR =  "E:/Datasets/mm-prod-search-and-reco" # Path.absolute(__file__).parent
PROD_DATA_DIR = f"{BASE_DATA_DIR}/data/prod"

client = chromadb.PersistentClient(path=f'{PROD_DATA_DIR}/chromadb')

In [12]:
collection = client.get_or_create_collection(name = "products")

In [17]:
product_data = d1.get_prod_data()

KeyboardInterrupt: 

In [ ]:
prod_docs = product_data[cols].apply(lambda x: " ".join(x), axis= 1)
prod_tfidf = tfidf.fit_transform(prod_docs)

prod_tfidf_embeddings = prod_tfidf.toarray()

In [ ]:
prod_docs

0        island of the blue dolphins island of  the blu...
1        the morning sidekick journal  habit tracker jo...
2        nasb giant print bible blue leathertex 2020 te...
3        learning without tears  printing power  2008 e...
4        bushnell falcon 10x50 wide angle binoculars bl...
                               ...                        
11145    vonanda faux leather sofa couch midcentury han...
11146    2021 amplified hd digital tv antenna long 180 ...
11147    65ft white christmas tree fraser fir pencil ar...
11148    womens sexy sequin dress formal gowns and even...
11149    black hoodie women oversized sweatshirt croppe...
Length: 11150, dtype: str

In [ ]:
BATCH_SIZE = 500

for i in tqdm(range(0, prod_tfidf.shape[0], BATCH_SIZE)):
    collection.add( ids = product_data["product_id"].iloc[i : i + BATCH_SIZE].to_list(),
                   embeddings= prod_tfidf_embeddings[i : i + BATCH_SIZE],
                   documents=prod_docs.iloc[i : i + BATCH_SIZE].to_list(),
                   metadatas=product_data[cols + ["image_url"]].iloc[i : i + BATCH_SIZE].to_dict("records"))
    

  0%|          | 0/23 [00:00<?, ?it/s]

100%|██████████| 23/23 [07:20<00:00, 19.16s/it]


In [13]:
print(collection.count())
print(collection.peek(1))

11150
{'ids': ['0547328613'], 'embeddings': array([[0., 0., 0., ..., 0., 0., 0.]], shape=(1, 10000)), 'documents': ['island of the blue dolphins island of  the blue dolphins use but like new gripping tale of karana’s survival strength and courage amidst vivid descriptions of island life\nrecommended for grades 25\n192 pages hmh books for young readers blue'], 'uris': None, 'included': ['metadatas', 'documents', 'embeddings'], 'data': None, 'metadatas': [{'product_title': 'island of the blue dolphins', 'product_color': 'blue', 'product_description': 'island of  the blue dolphins use but like new', 'product_brand': 'hmh books for young readers', 'image_url': 'https://m.media-amazon.com/images/I/81gSvYPU0xL._SY466_.jpg', 'product_bullet_point': 'gripping tale of karana’s survival strength and courage amidst vivid descriptions of island life\nrecommended for grades 25\n192 pages'}]}


In [20]:
query_tfdif_embedding = query_train_tfidf.toarray()
collection.query(query_embeddings=[query_tfdif_embedding[0].tolist()], n_results=10)

{'ids': [['B08L89ZGSR',
   'B07CZW7J7Z',
   'B078Y12Y6F',
   'B0777LR4ZX',
   'B07CPW4BVX',
   'B07C3C55WD',
   'B07TVJ1B2W',
   'B082YTZ4F6',
   'B01GKE7K7U',
   'B07PJL9W6D']],
 'embeddings': None,
 'documents': [['heavy duty motorcycle tool roll bag  tools not included  empty small tool bag and roll up wrench organizer tool pouch with elastic loops pocket molle webbing to wrap tool kit for mc bike atv utv ❌tools not included keep your tools safe  ready for battle · the mjölnir foldable tool roll bag from odinsberg is carefully designed by experienced motorcycle riders it’s the ideal tool kit accessory to bring on your motorbike adventure keep in your car truck or boat or just at home in your garage   · measuring 21x 95 53 x 24cm unfolded and 55 x 95 14 x 24cm folded it fits on most motorcycle tool boxes   · the 56 elastic loops have two sizes 075” and 14” to keep your wrenches sockets screwdrivers pliers and other tools measuring maximum 85” tight organized and ready for battle at a